# UCI operational EDA

Charts from the cleaned stay file. No CMS. No new raw-file edits.

SQL in notebook `04` already computed eligible `<30` rates. This notebook is the picture version of those same rules: **eligible stays only**, every rate has an **N**, small cells are not quoted.

**What this step is for**

An operations or quality lead can see which encounter traits in this 1999–2008 diabetes extract sit with a higher crude 30-day return rate, especially prior inpatient and ED use. They cannot treat this as a live EHR, and they cannot match a stay to a CMS hospital.

**Questions**

1. Which segments have the highest crude 30-day return rates?
2. How strongly does prior utilization relate to `<30` return?
3. What is associated with longer stays, as context only?

**Words used here**

- **Eligible stay.** Not death, hospice, still-in, or invalid gender. Rates and later the model use this set.
- **`<30` / `readmit_30`.** Came back within 30 days. `>30` is not this KPI.
- **Prior acute.** Inpatient + ED visits in the year before the stay. Locked bands: 0 / 1 / 2+.
- **Crude rate.** Count of `<30` in the segment / eligible stays in the segment. Not risk-adjusted.
- **Association.** Two things that move together. Not a cause.

**Steps**

1. Load `uci_encounter_mart.csv`. Keep eligible stays for rates.
2. Overall rate with N and an interval.
3. Chart prior-use bands, then age, admission type, diagnosis, and discharge (large cells only).
4. Length of stay as context, not a second target.
5. Leave a candidate feature list for notebook `07`.
6. Save figures to `figures/`.

This is not a CMS `READM_30_*` score.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display


In [2]:
def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "processed").is_dir():
            return path
    raise FileNotFoundError(
        "Could not find the project root. Run this notebook from the "
        "repository folder or from notebooks/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"

print(PROJECT_ROOT)
print(FIGURES_DIR)


C:\Users\Micaela\Documents\CODING\hospital-operations-analytics
C:\Users\Micaela\Documents\CODING\hospital-operations-analytics\figures


## Load eligible stays

IDs stay text. I keep the full file in memory so I can say how many stays were set aside. Every rate below uses `eligible == True`.


In [3]:
enc = pd.read_csv(
    PROCESSED_DATA_DIR / "uci_encounter_mart.csv",
    dtype={
        "encounter_id": "string",
        "patient_nbr": "string",
        "payer_code": "string",
    },
)
assert enc["encounter_id"].nunique() == len(enc) == 101766

eligible = enc.loc[enc["eligible_for_readmit"]].copy()
n_out = (~enc["eligible_for_readmit"]).sum()
print(f"stays: {len(enc):,}")
print(f"eligible: {len(eligible):,}")
print(f"set aside (cannot score): {n_out:,}")
print(f"<30 among eligible: {int(eligible['readmit_30'].sum()):,}")


def rate_table(frame: pd.DataFrame, by, min_n: int = 50) -> pd.DataFrame:
    g = (
        frame.groupby(by, dropna=False, observed=False)
        .agg(n_readmit=("readmit_30", "sum"), n_encounters=("readmit_30", "size"))
        .reset_index()
    )
    g = g.loc[g["n_encounters"] >= min_n].copy()
    p = g["n_readmit"] / g["n_encounters"]
    se = (p * (1 - p) / g["n_encounters"]) ** 0.5
    g["rate"] = p
    g["rate_lo"] = (p - 1.96 * se).clip(lower=0)
    g["rate_hi"] = (p + 1.96 * se).clip(upper=1)
    return g


def add_errorbars(ax, table: pd.DataFrame, x_col: str, order):
    pos = {label: i for i, label in enumerate(order)}
    for row in table.itertuples(index=False):
        x = getattr(row, x_col) if x_col != "index" else None
        if x not in pos:
            continue
        ax.errorbar(
            pos[x],
            row.rate,
            yerr=[[row.rate - row.rate_lo], [row.rate_hi - row.rate]],
            fmt="none",
            ecolor="black",
            capsize=3,
            elinewidth=1,
        )


stays: 101,766
eligible: 99,337
set aside (cannot score): 2,429
<30 among eligible: 11,312


## Overall eligible rate

11.4% is the number every segment chart should be read against. The interval is a normal approximation. It is not a model CI.


In [4]:
n = len(eligible)
k = int(eligible["readmit_30"].sum())
p = k / n
se = (p * (1 - p) / n) ** 0.5
print(f"{k:,} / {n:,} = {p:.3f} (95% interval {p - 1.96 * se:.3f} to {p + 1.96 * se:.3f})")
print("patients among eligible stays:", eligible["patient_nbr"].nunique())


11,312 / 99,337 = 0.114 (95% interval 0.112 to 0.116)
patients among eligible stays: 69985


## Prior utilization

This is the main operational cut. Bands are the locked 0 / 1 / 2+ on inpatient + ED in the year before the stay. Outpatient is shown separately. Higher prior use sitting with a higher `<30` rate is association, not proof that cutting ED visits would cut return.


In [5]:
acute = rate_table(eligible, "prior_acute_band")
acute = acute.sort_values("prior_acute_band")
display(acute)

fig, ax = plt.subplots(figsize=(6.4, 4.2))
order = ["0", "1", "2+"]
sns.barplot(data=acute, x="prior_acute_band", y="rate", order=order, color="#4C78A8", ax=ax)
add_errorbars(ax, acute, "prior_acute_band", order)
ax.axhline(p, color="#333333", linestyle="--", linewidth=1, label=f"eligible overall {p:.1%}")
ax.set_ylim(0, 0.28)
ax.set_xlabel("Prior acute visits (inpatient + ED, year before the stay)")
ax.set_ylabel("Crude 30-day return rate")
ax.set_title("Eligible <30 rate by prior acute use")
ax.legend(loc="upper left")
fig.savefig(FIGURES_DIR / "uci_rate_by_prior_acute.png")
plt.show()

cross = rate_table(eligible, ["inpatient_band", "emergency_band"], min_n=50)
cross_p = cross.pivot(index="inpatient_band", columns="emergency_band", values="rate")
print("inpatient x ED rate (n >= 50)")
display(cross.assign(n=cross["n_encounters"]).pivot_table(
    index="inpatient_band", columns="emergency_band", values=["n_encounters", "rate"]
))
outp = rate_table(eligible, "outpatient_band")
print("outpatient band")
display(outp)


,prior_acute_band,n_readmit,n_encounters,rate,rate_lo,rate_hi
0,0,5215,61736,0.084473,0.082279,0.086666
1,1,2424,19618,0.123560,0.118955,0.128165
2,2+,3673,17983,0.204248,0.198356,0.210141


inpatient x ED rate (n >= 50)


C:\Users\Micaela\AppData\Local\Temp\ipykernel_31256\4181081707.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


n_encounters                      rate                    
emergency_band            0       1      2+         0         1        2+
inpatient_band                                                           
0                   61736.0  3465.0  1039.0  0.084473  0.100433  0.121270
1                   16153.0  2000.0   830.0  0.128521  0.145000  0.179518
2+                  10354.0  2009.0  1751.0  0.206490  0.228472  0.291833

outpatient band


,outpatient_band,n_readmit,n_encounters,rate,rate_lo,rate_hi
0,0,9037,82988,0.108895,0.106776,0.111015
1,1,1186,8349,0.142053,0.134564,0.149541
2,2+,1089,8000,0.136125,0.128610,0.143640


## Age and admission type

Age is already a band in the file. Trauma Center and Newborn stay out of the admission chart (N under 50).


In [6]:
age = rate_table(eligible, "age")
age = age.sort_values("age")
display(age)

fig, ax = plt.subplots(figsize=(8.2, 4.2))
age_order = age["age"].tolist()
sns.barplot(data=age, x="age", y="rate", order=age_order, color="#4C78A8", ax=ax)
add_errorbars(ax, age, "age", age_order)
ax.axhline(p, color="#333333", linestyle="--", linewidth=1)
ax.set_ylim(0, 0.20)
ax.set_xlabel("Age band")
ax.set_ylabel("Crude 30-day return rate")
ax.set_title("Eligible <30 rate by age")
fig.savefig(FIGURES_DIR / "uci_rate_by_age.png")
plt.show()

adm = rate_table(eligible, "admission_type", min_n=50)
adm = adm.sort_values("n_encounters", ascending=False)
# label missing admission type
adm["admission_label"] = adm["admission_type"].fillna("(missing)")
display(adm)

fig, ax = plt.subplots(figsize=(6.8, 4.2))
adm_order = adm["admission_label"].tolist()
sns.barplot(data=adm, x="admission_label", y="rate", order=adm_order, color="#4C78A8", ax=ax)
# errorbars keyed on admission_label
adm_err = adm.rename(columns={"admission_label": "lab"})
pos = {label: i for i, label in enumerate(adm_order)}
for row in adm.itertuples(index=False):
    lab = row.admission_label
    ax.errorbar(
        pos[lab],
        row.rate,
        yerr=[[row.rate - row.rate_lo], [row.rate_hi - row.rate]],
        fmt="none",
        ecolor="black",
        capsize=3,
        elinewidth=1,
    )
ax.axhline(p, color="#333333", linestyle="--", linewidth=1)
ax.set_ylim(0, 0.18)
ax.set_xlabel("Admission type")
ax.set_ylabel("Crude 30-day return rate")
ax.set_title("Eligible <30 rate by admission type")
fig.savefig(FIGURES_DIR / "uci_rate_by_admission.png")
plt.show()


,age,n_readmit,n_encounters,rate,rate_lo,rate_hi
0,[0-10),3,160,0.018750,0.000000,0.039768
1,[10-20),40,690,0.057971,0.040534,0.075408
2,[20-30),236,1649,0.143117,0.126214,0.160020
3,[30-40),424,3764,0.112646,0.102546,0.122747
4,[40-50),1024,9607,0.106589,0.100418,0.112760
5,[50-60),1667,17060,0.097714,0.093258,0.102170
6,[60-70),2493,22058,0.113020,0.108842,0.117199
7,[70-80),3052,25327,0.120504,0.116494,0.124513
8,[80-90),2065,16434,0.125654,0.120586,0.130722
9,[90-100),308,2588,0.119011,0.106535,0.131486


C:\Users\Micaela\AppData\Local\Temp\ipykernel_31256\498463032.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,admission_type,n_readmit,n_encounters,rate,rate_lo,rate_hi,admission_label
1,Emergency,6194,52368,0.118278,0.115512,0.121044,Emergency
0,Elective,1958,18667,0.104891,0.100495,0.109287,Elective
4,Urgent,2056,18130,0.113403,0.108788,0.118019,Urgent
5,NaN,1103,10144,0.108734,0.102676,0.114792,(missing)


C:\Users\Micaela\AppData\Local\Temp\ipykernel_31256\498463032.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Diagnosis and discharge

Diagnosis is the ICD-9 chapter of `diag_1`. Discharge is known at the end of the stay. Death and hospice are already out. I only print cells with at least 200 stays so a 60-row transfer type cannot become the story.


In [7]:
dx = rate_table(eligible, "diag_1_group", min_n=200)
dx = dx.sort_values("n_encounters", ascending=False)
display(dx)

disch = rate_table(eligible, "discharge_disposition", min_n=200)
disch = disch.sort_values("rate", ascending=False)
disch["discharge_short"] = disch["discharge_disposition"].fillna("(missing)").str.slice(0, 48)
display(disch[["discharge_short", "n_readmit", "n_encounters", "rate", "rate_lo", "rate_hi"]])


,diag_1_group,n_readmit,n_encounters,rate,rate_lo,rate_hi
1,circulatory,3459,29583,0.116925,0.113264,0.120587
4,endocrine,1461,11304,0.129246,0.123062,0.135431
14,respiratory,1111,9918,0.112019,0.105811,0.118226
3,digestive,961,9072,0.105930,0.099597,0.112263
17,symptoms,683,7601,0.089857,0.083427,0.096286
8,injury,850,6851,0.124069,0.116263,0.131876
6,genitourinary,546,4963,0.110014,0.101308,0.118720
10,musculoskeletal,471,4935,0.095441,0.087243,0.103639
11,neoplasm,341,3131,0.108911,0.097999,0.119823
7,infectious,315,2553,0.123384,0.110627,0.136142


,discharge_short,n_readmit,n_encounters,rate,rate_lo,rate_hi
7,Discharged/transferred to another rehab fac incl,552,1992,0.277108,0.257453,0.296763
9,Discharged/transferred to another type of inpati,247,1184,0.208615,0.185470,0.231759
8,Discharged/transferred to another short term hos,342,2128,0.160714,0.145110,0.176319
3,Discharged/transferred to SNF,2046,13954,0.146625,0.140755,0.152494
16,Left AMA,90,623,0.144462,0.116856,0.172069
2,Discharged/transferred to ICF,104,815,0.127607,0.104700,0.150515
11,Discharged/transferred to home with home health,1638,12902,0.126957,0.121212,0.132702
18,(missing),551,4680,0.117735,0.108501,0.126969
1,Discharged to home,5602,60232,0.093007,0.090687,0.095327
5,Discharged/transferred to a long term care hospi,30,412,0.072816,0.047725,0.097906


## Length of stay (context)

`time_in_hospital` is 1–14 days by construction. I report median and the `<30` rate by day. Longer stays sit with a somewhat higher return rate through about day 8, then the pattern is noisier. This is not a second model target and it is not CMS ED minutes.


In [8]:
print(eligible.groupby("readmit_30")["time_in_hospital"].agg(["median", "mean", "count"]))
los = rate_table(eligible, "time_in_hospital", min_n=50)
los = los.sort_values("time_in_hospital")
display(los)

fig, ax = plt.subplots(figsize=(8.2, 4.2))
sns.barplot(data=los, x="time_in_hospital", y="rate", color="#4C78A8", ax=ax)
pos = {int(v): i for i, v in enumerate(los["time_in_hospital"])}
for row in los.itertuples(index=False):
    ax.errorbar(
        pos[int(row.time_in_hospital)],
        row.rate,
        yerr=[[row.rate - row.rate_lo], [row.rate_hi - row.rate]],
        fmt="none",
        ecolor="black",
        capsize=2,
        elinewidth=1,
    )
ax.axhline(p, color="#333333", linestyle="--", linewidth=1)
ax.set_ylim(0, 0.20)
ax.set_xlabel("Length of stay (days)")
ax.set_ylabel("Crude 30-day return rate")
ax.set_title("Eligible <30 rate by length of stay")
fig.savefig(FIGURES_DIR / "uci_rate_by_los.png")
plt.show()

print("median LOS by admission type")
display(
    eligible.groupby(eligible["admission_type"].fillna("(missing)"))["time_in_hospital"]
    .agg(["median", "mean", "count"])
    .sort_values("count", ascending=False)
)


            median      mean  count
readmit_30                         
0              4.0  4.329509  88025
1              4.0  4.767504  11312


,time_in_hospital,n_readmit,n_encounters,rate,rate_lo,rate_hi
0,1,1160,13821,0.083930,0.079307,0.088553
1,2,1704,16891,0.100882,0.096340,0.105424
2,3,1883,17432,0.108020,0.103412,0.112628
3,4,1640,13683,0.119857,0.114415,0.125299
4,5,1196,9749,0.122679,0.116167,0.129192
5,6,945,7354,0.128501,0.120853,0.136150
6,7,747,5696,0.131145,0.122378,0.139911
7,8,623,4270,0.145902,0.135313,0.156490
8,9,411,2879,0.142758,0.129979,0.155537
9,10,335,2262,0.148099,0.133461,0.162737


median LOS by admission type


C:\Users\Micaela\AppData\Local\Temp\ipykernel_31256\114710426.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,median,mean,count
admission_type,,,
Emergency,4.0,4.361843,52368
Elective,3.0,4.304548,18667
Urgent,4.0,4.595036,18130
(missing),3.0,4.221510,10144
Trauma Center,5.0,5.444444,18
Newborn,3.0,3.200000,10


## Light medication and lab cuts

A1C is missing on most stays (not tested). Med change and diabetes-med flags are associations only. I will not build the story on a drug name.


In [9]:
print("A1C")
display(rate_table(eligible, "A1Cresult", min_n=50))
print("med change")
display(rate_table(eligible, "change", min_n=50))
print("diabetes medication")
display(rate_table(eligible, "diabetesMed", min_n=50))
print("gender")
display(rate_table(eligible, "gender", min_n=50))
print("race")
display(rate_table(eligible, "race", min_n=50))


A1C


,A1Cresult,n_readmit,n_encounters,rate,rate_lo,rate_hi
0,>7,383,3775,0.101457,0.091825,0.111089
1,>8,809,8137,0.099422,0.092921,0.105924
2,Normal,481,4922,0.097725,0.089429,0.106020
3,NaN,9639,82503,0.116832,0.114640,0.119024


med change


,change,n_readmit,n_encounters,rate,rate_lo,rate_hi
0,No,5767,53217,0.108368,0.105727,0.111009
1,Yes,5545,46120,0.120230,0.117262,0.123198


diabetes medication


,diabetesMed,n_readmit,n_encounters,rate,rate_lo,rate_hi
0,No,2231,22620,0.09863,0.094744,0.102515
1,Yes,9081,76717,0.11837,0.116084,0.120656


gender


,gender,n_readmit,n_encounters,rate,rate_lo,rate_hi
0,Female,6128,53454,0.114641,0.111940,0.117341
1,Male,5184,45883,0.112983,0.110086,0.115880


race


,race,n_readmit,n_encounters,rate,rate_lo,rate_hi
0,African American,2149,18772,0.114479,0.109924,0.119034
1,Asian,65,628,0.103503,0.079678,0.127328
2,Caucasian,8554,74217,0.115257,0.112959,0.117554
3,Hispanic,212,2017,0.105107,0.091722,0.118491
4,Other,144,1471,0.097893,0.082706,0.113079
5,NaN,188,2232,0.084229,0.072707,0.095752


## Findings from this pass

Working notes. Crude rates. Eligible stays only. No CMS.

- **Overall.** 11,312 returns in 99,337 eligible stays, **11.4%** (about 11.2% to 11.6%). 2,429 stays were set aside and must stay out of the rate and the model.
- **Prior acute is the step that matters.** 0 visits: **8.4%** (5,215 / 61,736). 1 visit: **12.4%** (2,424 / 19,618). 2+: **20.4%** (3,673 / 17,983). Two-plus inpatient and two-plus ED together is about 29% on 1,751 stays. Outpatient 1 and 2+ are 14.2% and 13.6%, a smaller lift. This does not say that cutting ED visits would cut readmission.
- **Age is not a straight climb.** [20-30) is **14.3%** (236 / 1,649). [50-60) is 9.8%. [70-80) is 12.1% and [80-90) is 12.6%. [0-10) is 1.9% on 160 stays; I will not lead with it.
- **Admission.** Emergency 11.8% (6,194 / 52,368). Elective 10.5% (1,958 / 18,667). Urgent 11.3%. About 10,000 stays have a missing admission type (10.9%). Trauma/newborn stay unquoted.
- **Diagnosis.** Circulatory is the largest group (29,583, 11.7%). Endocrine 12.9%. Symptoms 9.0%. Crude chapters, not risk-adjusted.
- **Discharge (large cells).** Home 9.3% (5,602 / 60,232). SNF 14.7%. Home health 12.7%. Rehab 27.7% (552 / 1,992). Transfer to another inpatient 20.9%. Those destinations are known at discharge; they also mark a different pathway. Psych and "within this institution" cells are too small to quote.
- **LOS.** Median 4 days whether the stay returned or not. Mean is a bit higher for returns (4.8 vs 4.3). The `<30` rate rises from 8.4% at 1 day to about 14–15% around days 8–10, then bounces. Context, not a second target.
- **A1C / meds.** Most stays have no A1C. Tested groups sit slightly *below* the overall rate. Med change Yes 12.0% vs No 10.8%. Diabetes med Yes 11.8% vs No 9.9%. Associations only. Race gaps are small; missing race is 8.4% on 2,232 stays (do not treat missing as a real group).

Figures: `figures/uci_rate_by_prior_acute.png`, `figures/uci_rate_by_age.png`, `figures/uci_rate_by_admission.png`, `figures/uci_rate_by_los.png`.

This is not a CMS heart-failure readmission score.


## Candidate features for notebook `07`

Split by `patient_nbr`, not by encounter. Use eligible stays only. Fields should be known at or before the discharge-time decision.

**Use**

- Demographics: `age`, `gender`, `race` (keep missing as missing)
- Arrival: `admission_type`, `admission_source`
- Prior use: `number_inpatient`, `number_emergency`, `number_outpatient` (or the 0 / 1 / 2+ bands). Prior acute is the strongest cut here.
- Stay burden: `time_in_hospital`, `num_lab_procedures`, `num_procedures`, `num_medications`, `number_diagnoses`
- Clinical: `diag_1_group`, `A1Cresult`, `max_glu_serum`
- Meds as flags, not a 20-drug story: `change`, `diabetesMed`, `insulin`

**Use with a flag**

- `discharge_disposition` is known at discharge. Rehab and other-inpatient transfers already sit with high crude rates. Include only if the model is scored at discharge, and say so.

**Do not use**

- `readmitted`, `readmit_30` (target)
- `cannot_return`, `eligible_for_readmit` (filter, not a feature)
- `encounter_id`
- Raw `diag_1` / `diag_2` / `diag_3` as high-cardinality codes (the chapter is enough for an interpretable model)
- Weight (dropped; ~97% missing)

Next: notebook `07`, triage model. Catch rate vs follow-up workload, not accuracy.


In [10]:
print("saved")
for path in sorted(FIGURES_DIR.glob("uci_*.png")):
    print(path.name, path.stat().st_size)


saved
uci_rate_by_admission.png 38008
uci_rate_by_age.png 44096
uci_rate_by_los.png 40484
uci_rate_by_prior_acute.png 35100
